# DistilBERT Model Evaluation

Evaluate the DistilBERT transaction classifier across 8 serving strategies:
1. PyTorch baseline (CPU)
2. ONNX (no optimization)
3. ONNX with graph optimization
4. Dynamic quantization (Intel Neural Compressor)
5. Static quantization - aggressive
6. Static quantization - conservative
7. GPU execution providers (CUDA, TensorRT)
8. OpenVINO execution provider

**Metrics**: model size, inference latency (p50/p95/p99), single-sample throughput, batch throughput

In [ ]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))
from benchmark_utils import (
    get_model_size_mb, benchmark_latency, benchmark_batch_throughput,
    benchmark_ort_session, print_benchmark_results, collect_result_row
)

MODEL_NAME = "distilbert"
MODEL_DIR = f"models/{MODEL_NAME}"
ONNX_DIR = f"models/{MODEL_NAME}_onnx"
os.makedirs(ONNX_DIR, exist_ok=True)

all_results = []

## Load sample data for benchmarking

In [ ]:
sample_texts = [
    "TESCO STORES 3456 -45.20 Friday",
    "DOMINOS #7548 -22.50 Thursday",
    "OLIVE GARDEN #9785 -67.30 Saturday",
    "AMAZON.COM -129.99 Monday",
    "SHELL OIL 12345 -55.00 Tuesday",
    "NETFLIX.COM -15.99 Wednesday",
    "WHOLE FOODS MKT -88.45 Sunday",
    "UBER *TRIP -18.75 Friday",
]

single_text = sample_texts[0]
batch_texts = sample_texts * 4  # 32 samples for batch benchmark

print(f"Single sample: '{single_text}'")
print(f"Batch size: {len(batch_texts)}")

## 1. PyTorch Baseline (CPU)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()
device = torch.device("cpu")
model.to(device)

model_size = get_model_size_mb(MODEL_DIR)
print(f"Model size: {model_size:.2f} MB")

In [ ]:
def predict_pytorch_single(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits

def predict_pytorch_batch(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits

latency_results = benchmark_latency(predict_pytorch_single, single_text)
batch_results = benchmark_batch_throughput(predict_pytorch_batch, batch_texts)
results_pytorch = {**latency_results, **batch_results}

print_benchmark_results(results_pytorch, MODEL_NAME, "pytorch_cpu", model_size)
all_results.append(collect_result_row(MODEL_NAME, "pytorch_cpu", "CPU", model_size, results_pytorch))

## 2. ONNX Export (No Optimization)

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from optimum.exporters.onnx import main_export

onnx_model_path = os.path.join(ONNX_DIR, "model.onnx")

if not os.path.exists(onnx_model_path):
    main_export(
        MODEL_DIR,
        output=ONNX_DIR,
        task="text-classification",
        opset=17,
    )
    print(f"ONNX model exported to {ONNX_DIR}")
else:
    print(f"ONNX model already exists at {onnx_model_path}")

onnx_size = get_model_size_mb(onnx_model_path)
print(f"ONNX model size: {onnx_size:.2f} MB")

In [ ]:
import onnxruntime as ort

session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

ort_session_no_opt = ort.InferenceSession(
    onnx_model_path,
    sess_options=session_options,
    providers=['CPUExecutionProvider']
)

print(f"Providers: {ort_session_no_opt.get_providers()}")

def get_ort_inputs(texts, is_batch=False):
    if isinstance(texts, str):
        texts = [texts]
    encoded = tokenizer(texts, return_tensors="np", padding=True, truncation=True, max_length=64)
    return {k: v for k, v in encoded.items() if k in [inp.name for inp in ort_session_no_opt.get_inputs()]}

sample_ort = get_ort_inputs(single_text)
batch_ort = get_ort_inputs(batch_texts)

results_onnx_no_opt = benchmark_ort_session(ort_session_no_opt, sample_ort, batch_ort)
print_benchmark_results(results_onnx_no_opt, MODEL_NAME, "onnx_no_opt", onnx_size)
all_results.append(collect_result_row(MODEL_NAME, "onnx_no_opt", "CPU", onnx_size, results_onnx_no_opt))

## 3. ONNX with Graph Optimization (ORT_ENABLE_EXTENDED)

In [ ]:
optimized_model_path = os.path.join(ONNX_DIR, "model_optimized.onnx")

session_options_opt = ort.SessionOptions()
session_options_opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
session_options_opt.optimized_model_filepath = optimized_model_path

ort_session_opt = ort.InferenceSession(
    onnx_model_path,
    sess_options=session_options_opt,
    providers=['CPUExecutionProvider']
)

opt_size = get_model_size_mb(optimized_model_path) if os.path.exists(optimized_model_path) else onnx_size
print(f"Optimized ONNX model size: {opt_size:.2f} MB")

ort_session_opt_loaded = ort.InferenceSession(
    optimized_model_path if os.path.exists(optimized_model_path) else onnx_model_path,
    providers=['CPUExecutionProvider']
)

results_onnx_opt = benchmark_ort_session(ort_session_opt_loaded, sample_ort, batch_ort)
print_benchmark_results(results_onnx_opt, MODEL_NAME, "onnx_optimized", opt_size)
all_results.append(collect_result_row(MODEL_NAME, "onnx_optimized", "CPU", opt_size, results_onnx_opt))

## 4. Dynamic Quantization (Intel Neural Compressor)

In [ ]:
import neural_compressor
from neural_compressor import quantization

fp32_model = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

config_dynamic = neural_compressor.PostTrainingQuantConfig(
    approach="dynamic"
)

q_model_dynamic = quantization.fit(
    model=fp32_model,
    conf=config_dynamic
)

dynamic_quant_path = os.path.join(ONNX_DIR, "model_quantized_dynamic.onnx")
q_model_dynamic.save_model_to_file(dynamic_quant_path)

dq_size = get_model_size_mb(dynamic_quant_path)
print(f"Dynamic quantized model size: {dq_size:.2f} MB")

In [ ]:
ort_session_dq = ort.InferenceSession(
    dynamic_quant_path,
    providers=['CPUExecutionProvider']
)

results_dq = benchmark_ort_session(ort_session_dq, sample_ort, batch_ort)
print_benchmark_results(results_dq, MODEL_NAME, "dynamic_quant", dq_size)
all_results.append(collect_result_row(MODEL_NAME, "dynamic_quant", "CPU", dq_size, results_dq))

## 5. Static Quantization - Aggressive

Uses `quant_level=1` with `tolerable_loss=0.05` and calibration data from transaction payee text.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PayeeCalibrationDataset(Dataset):
    """Calibration dataset from transaction payee text."""
    def __init__(self, csv_path, tokenizer, max_samples=256, max_length=64):
        df = pd.read_csv(csv_path)
        self.texts = df['payee'].dropna().unique()[:max_samples].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='np'
        )
        input_feed = {k: v.squeeze(0) for k, v in encoded.items()
                      if k in ['input_ids', 'attention_mask']}
        return (input_feed, 0)  # dummy label

calib_dataset = PayeeCalibrationDataset('data/transactions.csv', tokenizer)
calib_dataloader = neural_compressor.data.DataLoader(
    framework='onnxruntime',
    dataset=calib_dataset,
    batch_size=1
)
print(f"Calibration dataset size: {len(calib_dataset)}")

In [ ]:
fp32_model_agg = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

config_static_aggressive = neural_compressor.PostTrainingQuantConfig(
    accuracy_criterion=neural_compressor.config.AccuracyCriterion(
        criterion="absolute",
        tolerable_loss=0.05
    ),
    approach="static",
    device='cpu',
    quant_level=1,
    quant_format="QOperator",
    recipes={"graph_optimization_level": "ENABLE_EXTENDED"},
    calibration_sampling_size=128
)

q_model_agg = quantization.fit(
    model=fp32_model_agg,
    conf=config_static_aggressive,
    calib_dataloader=calib_dataloader,
    eval_dataloader=calib_dataloader
)

agg_quant_path = os.path.join(ONNX_DIR, "model_quantized_aggressive.onnx")
q_model_agg.save_model_to_file(agg_quant_path)

agg_size = get_model_size_mb(agg_quant_path)
print(f"Aggressive quantized model size: {agg_size:.2f} MB")

In [ ]:
ort_session_agg = ort.InferenceSession(
    agg_quant_path,
    providers=['CPUExecutionProvider']
)

results_agg = benchmark_ort_session(ort_session_agg, sample_ort, batch_ort)
print_benchmark_results(results_agg, MODEL_NAME, "static_quant_aggressive", agg_size)
all_results.append(collect_result_row(MODEL_NAME, "static_quant_aggressive", "CPU", agg_size, results_agg))

## 6. Static Quantization - Conservative

Uses `quant_level=0` with `tolerable_loss=0.01` for a less aggressive quantization.

In [ ]:
fp32_model_con = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

config_static_conservative = neural_compressor.PostTrainingQuantConfig(
    accuracy_criterion=neural_compressor.config.AccuracyCriterion(
        criterion="absolute",
        tolerable_loss=0.01
    ),
    approach="static",
    device='cpu',
    quant_level=0,
    quant_format="QOperator",
    recipes={"graph_optimization_level": "ENABLE_EXTENDED"},
    calibration_sampling_size=128
)

q_model_con = quantization.fit(
    model=fp32_model_con,
    conf=config_static_conservative,
    calib_dataloader=calib_dataloader,
    eval_dataloader=calib_dataloader
)

con_quant_path = os.path.join(ONNX_DIR, "model_quantized_conservative.onnx")
q_model_con.save_model_to_file(con_quant_path)

con_size = get_model_size_mb(con_quant_path)
print(f"Conservative quantized model size: {con_size:.2f} MB")

In [ ]:
ort_session_con = ort.InferenceSession(
    con_quant_path,
    providers=['CPUExecutionProvider']
)

results_con = benchmark_ort_session(ort_session_con, sample_ort, batch_ort)
print_benchmark_results(results_con, MODEL_NAME, "static_quant_conservative", con_size)
all_results.append(collect_result_row(MODEL_NAME, "static_quant_conservative", "CPU", con_size, results_con))

## 7. GPU Execution Providers

**Note**: Run this section using the `jupyter-eval-gpu` container (Dockerfile.jupyter-eval-gpu) which has `onnxruntime-gpu` installed.

### 7a. CUDAExecutionProvider

In [ ]:
try:
    ort_session_cuda = ort.InferenceSession(
        onnx_model_path,
        providers=['CUDAExecutionProvider']
    )
    print(f"CUDA providers: {ort_session_cuda.get_providers()}")

    results_cuda = benchmark_ort_session(ort_session_cuda, sample_ort, batch_ort)
    print_benchmark_results(results_cuda, MODEL_NAME, "onnx_cuda", onnx_size)
    all_results.append(collect_result_row(MODEL_NAME, "onnx_cuda", "RTX 6000", onnx_size, results_cuda))
except Exception as e:
    print(f"CUDAExecutionProvider not available: {e}")
    print("Run this cell in the GPU container (jupyter-eval-gpu).")

### 7b. TensorrtExecutionProvider

In [ ]:
try:
    ort_session_trt = ort.InferenceSession(
        onnx_model_path,
        providers=['TensorrtExecutionProvider']
    )
    print(f"TensorRT providers: {ort_session_trt.get_providers()}")

    results_trt = benchmark_ort_session(ort_session_trt, sample_ort, batch_ort)
    print_benchmark_results(results_trt, MODEL_NAME, "onnx_tensorrt", onnx_size)
    all_results.append(collect_result_row(MODEL_NAME, "onnx_tensorrt", "RTX 6000", onnx_size, results_trt))
except Exception as e:
    print(f"TensorrtExecutionProvider not available: {e}")
    print("Run this cell in the GPU container (jupyter-eval-gpu).")

## 8. OpenVINO Execution Provider

**Note**: Run this section using the `jupyter-eval-openvino` container (Dockerfile.jupyter-eval-openvino).

In [ ]:
try:
    ort_session_ov = ort.InferenceSession(
        onnx_model_path,
        providers=['OpenVINOExecutionProvider']
    )
    print(f"OpenVINO providers: {ort_session_ov.get_providers()}")

    results_ov = benchmark_ort_session(ort_session_ov, sample_ort, batch_ort)
    print_benchmark_results(results_ov, MODEL_NAME, "onnx_openvino", onnx_size)
    all_results.append(collect_result_row(MODEL_NAME, "onnx_openvino", "CPU (OpenVINO)", onnx_size, results_ov))
except Exception as e:
    print(f"OpenVINOExecutionProvider not available: {e}")
    print("Run this cell in the OpenVINO container (jupyter-eval-openvino).")

## Summary

In [ ]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))
results_df.to_csv(f"results/{MODEL_NAME}_evaluation.csv", index=False)
print(f"\nResults saved to results/{MODEL_NAME}_evaluation.csv")